In [1]:
import json
import os
import httpx2
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key and openrouter_api_key.startswith("sk-or-v1-"):
    print(f"OpenRouter API key exists and begins {openrouter_api_key[:9]}")
else:
    print("No OpenRouter API key found. Please check your .env file (Setup part 4).")

OpenRouter API key exists and begins sk-or-v1-


In [3]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "openrouter/free"

# Retry up to 5 times when a free model is busy
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

In [4]:
messages = [{"role": "user", "content": "Hi there!"}]

response = openrouter.chat.completions.create(model=MODEL, messages=messages)

print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Answered by dots-studio/dots-3-note-preview:free




Hi there! I'm dots, an AI model developed by Dots Studio at rednote. How can I help you today?

In [5]:
MAZE_URL = "https://maze.edwarddonner.com"

def read_web_page(url):
    """Read a web page in the maze and return the text on the page, without the HTML"""
    print(f"Reading web page: {url}")
    if not url.startswith(MAZE_URL):
        return f"Sorry, this tool can only read pages in the maze. Please ask for a URL that starts with {MAZE_URL}"
    response = httpx2.get(url, timeout=20)
    page = BeautifulSoup(response.text, "html.parser")
    return page.body.get_text(separator="\n", strip=True)

In [6]:
print(read_web_page("https://maze.edwarddonner.com"))

Reading web page: https://maze.edwarddonner.com
The Maze
This page is for an AI that can read web pages. Your goal is to find the way out of a small maze and bring back the message that is waiting there. Every location in the maze is a web page that describes the room and lists its exits.
Step 1 is the entrance:
Location 1
https://maze.edwarddonner.com/1
Start with the maze now. Open the entrance, then follow the exits one page at a time. Some passages are dead ends, so go back and try another way when you reach one.
Keep the person you are working with updated as you go. After each location, say where you are and what you found there, and show a mini-map of the maze as you have seen it so far, drawn in text with the rooms you have visited and the exits you know about. When you reach the way out, give the message and a final map of your whole journey.
If you are a person reading this, give this address to ChatGPT or Claude and ask it to do what the page says.


In [7]:
print(read_web_page("https://edwarddonner.com"))

Reading web page: https://edwarddonner.com
Sorry, this tool can only read pages in the maze. Please ask for a URL that starts with https://maze.edwarddonner.com


In [8]:
read_web_page_json = {
    "type": "function",
    "function": {
        "name": "read_web_page",
        "description": "Read a web page and return the text on the page",
        "parameters": {
            "type": "object",
            "properties": {
                "url": {"type": "string", "description": "The full URL of the web page to request"},
            },
            "required": ["url"],
        },
    },
}

tools = [read_web_page_json]

In [9]:
def handle_tool_call(tool_call):
    """Run the tool that the model asked for and package the result as a message"""
    try:
        arguments = json.loads(tool_call.function.arguments)
        result = read_web_page(**arguments)
    except Exception as error:
        result = f"The tool call failed: {error}"
    return {"role": "tool", "tool_call_id": tool_call.id, "content": result}

In [10]:
messages = [{"role": "user", "content": "Please read https://maze.edwarddonner.com and tell me what it says. Just describe the page for now, without going into the maze."}]

response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
message = response.choices[0].message

print(f"Answered by {response.model}")
print(f"Finish reason: {response.choices[0].finish_reason}")
print(message.tool_calls)

Answered by poolside/laguna-xs-2.1:free
Finish reason: tool_calls
[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-5a888600761b4ce499b2e8432162b79c', function=Function(arguments='{"url": "https://maze.edwarddonner.com"}', name='read_web_page'), type='function', index=0)]


In [11]:
messages.append(message)
for tool_call in message.tool_calls:
    messages.append(handle_tool_call(tool_call))

response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)

print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Reading web page: https://maze.edwarddonner.com
Answered by liquid/lfm-2.5-2.6b:free


The page at https://maze.edwarddonner.com is an interactive maze-solving challenge. Here's what it says:

**Title:** The Maze

**Description:** This is a puzzle where you control an AI that must navigate through a small maze by visiting web pages (each representing a room). Your goal is to find the exit and retrieve the message hidden somewhere in the maze.

**How to play:**
- **Step 1:** Start at the entrance (Location 1) at `https://maze.edwarddonner.com/1`
- Each location is a web page describing the room and listing its available exits
- You must explore one page at a time, following exits
- Be careful—some passages are dead ends; backtrack if you hit one
- Keep track of your progress and update the player (me) after each location
- Show a text-based mini-map of all visited rooms and known exits
- When you reach the exit, deliver the message and present a final map of your entire journey

**In short:** This is the starting instruction screen. The actual maze begins at `/1`, and from there you'll encounter individual room pages with their own descriptions and exit options. Would you like me to proceed to Location 1 and start exploring the maze?

Part 2 - Harness Loops  

In [12]:
SYSTEM_PROMPT = """
You are an agent that completes tasks by using your tools, one step at a time.
Your replies are displayed as Markdown, so format them in Markdown with headings, bold text and lists.
Put any map or diagram in a code block so that it lines up. Do not use emojis.
"""

In [13]:
def run_agent(task, max_steps=25):
    """Call the model in a loop, running the tool whenever it asks, until it gives a final answer"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    for step in range(1, max_steps + 1):
        response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        message = response.choices[0].message
        print(f"Step {step} answered by {response.model}")

        text = (message.content or "").strip()
        if text:
            display(Markdown(text))

        if message.tool_calls:
            messages.append(message)
            for tool_call in message.tool_calls:
                messages.append(handle_tool_call(tool_call))
        elif text:
            return
        else:
            print("The reply was empty, so we ask again")

    print("The agent ran out of steps")

In [14]:
run_agent("Have a look at https://maze.edwarddonner.com and do what it says.")

Step 1 answered by liquid/lfm-2.5-2.6b:free
Reading web page: https://maze.edwarddonner.com
Step 2 answered by cohere/north-mini-code:free
Reading web page: https://maze.edwarddonner.com/1
Step 3 answered by nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free
Reading web page: https://maze.edwarddonner.com/14
Step 4 answered by nvidia/nemotron-3-ultra-550b-a55b:free
Reading web page: https://maze.edwarddonner.com/62
Step 5 answered by poolside/laguna-s-2.1:free
Reading web page: https://maze.edwarddonner.com/39
Step 6 answered by poolside/laguna-xs-2.1:free
Reading web page: https://maze.edwarddonner.com/71
Step 7 answered by dots-studio/dots-3-note-preview:free
Reading web page: https://maze.edwarddonner.com/62
Step 8 answered by cohere/north-mini-code:free
Reading web page: https://maze.edwarddonner.com/48
Step 9 answered by dots-studio/dots-3-note-preview:free
Reading web page: https://maze.edwarddonner.com/83
Step 10 answered by minimax/minimax-m3.1-flash:free
Reading web page: http

I've successfully navigated through the maze and found the exit! Here's a summary of my journey:

## Journey Summary

**Route taken:**
- Location 1 (entrance) → North → Location 14 → North → Location 62 → East → Location 48 → East → Location 83 → North → Location 25 → East → Location 35 (**EXIT**)

**Message found at the exit:**
> "Welcome to the start of the course! Prepare for 3 intensive weeks ahead."

## Maze Map

```
                    [27]  ← unexplored (East from 1)
                     ↑
    [1] ──East──→ [14] ──North──→ [62] ──North──→ [39] ──East──→ [71] (dead end)
   Entrance        ↑                ↑                ↑            ↑
                   │                │                │            └──South── back to [39]
                   │                │                │
                   └──South─────────┘                └──East──→ [48] ──East──→ [83]
                                                            ↑        ↑          │
                                                            │        │          │North
                                                            │        └──West────┘          │
                                                            │                             │East
                                                            │                        [25] ←───North
                                                            │                          ↓  │
                                                            │                     Daylight│South
                                                            │                         [35] → EXIT!
                                                            │
                                                            └──────────────────────────────
                                                                  
```

**Key:**
- **[1]** - Entrance
- **[35]** - Exit (daylight)
- **[71]** - Dead end
- **[27]** - Unexplored branch (east from entrance)

**Dead ends encountered:**
- Location 71 (a cupboard with a broom)
- Location 39 only led back or to the dead end at 71

**Final path out:** 1 → 14 → 62 → 48 → 83 → 25 → 35 → Out